# GPU Usage

For large images and large kernel sizes, the featurisation step can be slow. One solution is GPU-powered featurisation, which are pretty fast at doing large convolutions (which most our blurs, edge detectors and membrane projections are). `isb` supports both GPU-enabled featurisation via `pytorch` and GPU-enabled XGBoost predictions via `CuPy`. This notebook shows how you can get that working and gives some indication of the speedup.

In [ ]:
from interactive_seg_backend.file_handling import load_image
from interactive_seg_backend import featurise, TrainingConfig, FeatureConfig

We do of course need to have pytorch installed for this to work:

In [ ]:
from interactive_seg_backend.features.gpu_utils import TORCH_AVAILABLE, CUPY_AVAILABLE
print(f"Torch installed: {TORCH_AVAILABLE}; CuPy installed: {CUPY_AVAILABLE}")

Let's use some relatively big (but not enormous!) kernels

In [ ]:
cpu_feat_cfg = FeatureConfig(
    gaussian_blur=True,
    sobel_filter=True,
    hessian_filter=True,
    difference_of_gaussians=True,
    membrane_projections=True,
    mean=True,
    sigmas=(8, 16, 32),
    cast_to='f16')
cpu_train_cfg = TrainingConfig(feature_config=cpu_feat_cfg, classifier='xgb')

gpu_feat_cfg = FeatureConfig(
    gaussian_blur=True,
    sobel_filter=True,
    hessian_filter=True,
    difference_of_gaussians=True,
    membrane_projections=True,
    mean=True,
    sigmas=(8, 16, 32),
    cast_to='f16')
gpu_train_cfg = TrainingConfig(feature_config=gpu_feat_cfg, classifier='xgb', use_gpu=True)

And make the image a bit bigger

In [ ]:
from skimage.transform import resize
import numpy as np

SF = 3
image = load_image('../tests/data/0.tif')
image_large = resize(image, (512 * SF, 512 * SF)).astype(np.float16)

In [ ]:
from time import time

cpu_start = time()
features_cpu = featurise(image_large, cpu_train_cfg)
cpu_end = time()
cpu_feat_time = cpu_end - cpu_start
print(f"CPU featurisation took {cpu_feat_time:.2f} seconds")

In [ ]:
from torch.cuda import synchronize

synchronize()
gpu_start = time()
features_gpu = featurise(image_large, gpu_train_cfg)
synchronize()
gpu_end = time()
gpu_feat_time = gpu_end - gpu_start
print(f"GPU featurisation took {gpu_feat_time:.2f} seconds")

In [ ]:
from interactive_seg_backend import get_model, train_and_apply
from interactive_seg_backend.file_handling import load_labels

labels = load_labels('../tests/data/0_labels.tif')
labels_large = resize(labels, (512 * SF, 512 * SF), order=0, preserve_range=True).astype(np.uint8)
xgb_cpu = get_model('xgb', {}, to_gpu=False)
xgb_gpu = get_model('xgb', {}, to_gpu=True)

In [ ]:
cpu_start = time()
pred_cpu, _, _ = train_and_apply(features_cpu, labels_large, cpu_train_cfg)
cpu_end = time()
cpu_train_apply_time = cpu_end - cpu_start
print(f"CPU train  & apply took {cpu_train_apply_time:.2f} seconds")

In [ ]:
gpu_start = time()
pred_gpu, _, _= train_and_apply(features_gpu, labels_large, gpu_train_cfg)
gpu_end = time()
gpu_train_apply_time = gpu_end - gpu_start
print(f"GPU train  & apply took {gpu_train_apply_time:.2f} seconds")

In [ ]:
import matplotlib.pyplot as plt
from interactive_seg_backend.utils import to_rgb_arr, apply_labels_as_overlay, recolor_labels, PALETTE_RGB_NORM
fig, axs = plt.subplots(1, 3, figsize=(9, 3))

cpu_time_title = rf"$t_{{feat}}={cpu_feat_time:.2f}s$; $t_{{fit+apply}}={cpu_train_apply_time:.2f}s$"
gpu_time_title = rf"$t_{{feat}}={gpu_feat_time:.2f}s$; $t_{{fit+apply}}={gpu_train_apply_time:.2f}s$"


titles = [f"Image \n {(image_large.shape)}", "CPU Prediction \n" + cpu_time_title, "GPU Prediction \n" + gpu_time_title]
img_rgb = to_rgb_arr((255 * image_large).astype(np.uint8))
img_with_labels = apply_labels_as_overlay(labels_large, img_rgb, PALETTE_RGB_NORM[1:], )
arrs = [img_with_labels, recolor_labels(pred_cpu + 1), recolor_labels(pred_gpu + 1)]

for ax, arr, title in zip(axs, arrs, titles):
    ax.imshow(arr)
    ax.set_title(title)
    ax.axis('off')
